# Pilot C Resume Train

?? Pilot C run id? ???? ?? run ??? ?? `checkpoint-*`?? ??? ????.

?? `# 2) Setup` ?? `RESUME_RUN_ID`? ?? run id? ?? ? ????. ? run? ??? ???.


In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_pilot_c_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


In [ ]:
# 2) Setup: Drive, data, model cache, run paths
from google.colab import drive
from pathlib import Path
import os
import shutil

drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import random
import re
import subprocess
import zipfile
from datetime import datetime

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"
LOCAL_MODEL_DIR = Path("/content/Qwen2-VL-7B-Instruct")


def print_runtime_storage():
    print("Storage check for /content:")
    try:
        subprocess.run(["df", "-h", "/content"], check=False)
    except Exception as exc:
        total, used, free = shutil.disk_usage("/content")
        print(f"/content free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB ({exc})")

    try:
        meminfo = {}
        with open("/proc/meminfo", "r", encoding="utf-8") as handle:
            for line in handle:
                key, value = line.split(":", 1)
                meminfo[key] = int(value.strip().split()[0]) / (1024 ** 2)
        print(f"RAM available: {meminfo.get('MemAvailable', 0):.1f} GB / total: {meminfo.get('MemTotal', 0):.1f} GB")
    except Exception as exc:
        print("RAM check skipped:", exc)

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB")


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def copy_drive_cache_to_local():
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        raise FileNotFoundError(f"Drive model cache is incomplete or missing: {DRIVE_MODEL_DIR}")

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using existing local model:", LOCAL_MODEL_DIR)
        return

    print("Copying base model from Drive to Colab local disk...")
    print("  from:", DRIVE_MODEL_DIR)
    print("  to  :", LOCAL_MODEL_DIR)
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR


def download_base_model_to_local_and_cache():
    print("Base model cache not found. Downloading with Hugging Face snapshot_download to local disk first:")
    print("repo:", MODEL_REPO_ID)
    if LOCAL_MODEL_DIR.exists() and not model_cache_is_complete(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    from huggingface_hub import snapshot_download
    hf_token = os.environ.get("HF_TOKEN")
    model_dir = snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        token=hf_token,
        max_workers=8,
    )
    print("Downloaded:", model_dir)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR

    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(LOCAL_MODEL_DIR, tmp)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)
    print("Saved to Drive:", DRIVE_MODEL_DIR)


def ensure_base_model_path():
    print_runtime_storage()

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using local model:", LOCAL_MODEL_DIR)
    elif model_cache_is_complete(DRIVE_MODEL_DIR):
        copy_drive_cache_to_local()
    elif not USE_MODELSCOPE_BASE_MODEL:
        download_base_model_to_local_and_cache()
    else:
        print("Base model cache not found. Downloading via ModelScope:", MODEL_REPO_ID)
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        if LOCAL_MODEL_DIR.exists():
            shutil.rmtree(LOCAL_MODEL_DIR)
        shutil.copytree(model_dir, LOCAL_MODEL_DIR)
        assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR
        DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(LOCAL_MODEL_DIR, tmp)
        if DRIVE_MODEL_DIR.exists():
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
        print("Saved to Drive:", DRIVE_MODEL_DIR)

    print_runtime_storage()
    print("Using local model:", LOCAL_MODEL_DIR)
    return str(LOCAL_MODEL_DIR)


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"

# Required: set this to the run id you want to resume, for example "20260721_012345".
RESUME_RUN_ID = None
if RESUME_RUN_ID is None:
    raise ValueError("Set RESUME_RUN_ID to an existing run id before running this notebook.")

RUN_ID = str(RESUME_RUN_ID)
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID
OUTPUT_DIR = RUN_ROOT / "multitask_bipair_conditional"
EVAL_DIR = OUTPUT_DIR / "eval"
BEST_ADAPTER_DIR = OUTPUT_DIR / "best_adapter"
assert RUN_ROOT.is_dir(), RUN_ROOT
assert OUTPUT_DIR.is_dir(), OUTPUT_DIR
EVAL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None
QUICK_EVAL_ROWS = 50
RUN_QUICK_EVAL_DURING_TRAIN = True

TASK_RATIOS = {
    "order": 0.30,
    "pairwise": 0.25,
    "first": 0.10,
    "last": 0.10,
    "fixed_first": 0.10,
    "fixed_last": 0.10,
    "fixed_endpoints": 0.05,
}
TASK_LOSS_WEIGHTS = {task: 1.0 for task in TASK_RATIOS}

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
MAX_TRAIN_STEPS = -1
SAVE_STEPS = 100
LOGGING_STEPS = 20

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("output:", OUTPUT_DIR)
print("model:", MODEL_ID)


In [ ]:
# 3) Data split and Pilot C multitask record generation
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def parse_compact_order(text, expected_len=4):
    values = [int(x) for x in re.findall(r"[1-4]", str(text))]
    if len(values) != expected_len or len(set(values)) != expected_len:
        return None
    return values


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def pair_target_for_order(order, a, b):
    ranks = {frame: idx for idx, frame in enumerate(order)}
    return "A" if ranks[int(a)] < ranks[int(b)] else "B"


def build_pair_groups(base):
    groups = []
    order = base["order"]
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        group = []
        for left, right in [(a, b), (b, a)]:
            item = copy.deepcopy(base)
            item.update({
                "task_type": "pairwise",
                "pair": [left, right],
                "image_paths": [base["image_paths"][left - 1], base["image_paths"][right - 1]],
                "target": pair_target_for_order(order, left, right),
            })
            group.append(item)
        groups.append(group)
    return groups


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    pools["pairwise_groups"] = []
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        pools["last"].append(item)

        pair_groups = build_pair_groups(base)
        pools["pairwise_groups"].extend(pair_groups)
        for group in pair_groups:
            pools["pairwise"].extend(group)

        first = order[0]
        remaining = [x for x in order if x != first]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order(remaining)})
        pools["fixed_first"].append(item)

        last = order[-1]
        remaining = [x for x in order if x != last]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order(remaining)})
        pools["fixed_last"].append(item)

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        pools["fixed_endpoints"].append(item)
    return pools


def sample_records(records, count, rng):
    records = list(records)
    if count <= len(records):
        indices = rng.choice(len(records), size=count, replace=False)
    else:
        base_indices = np.arange(len(records))
        extra_indices = rng.choice(len(records), size=count - len(records), replace=True)
        indices = np.concatenate([base_indices, extra_indices])
        rng.shuffle(indices)
    return [records[int(index)] for index in indices]


def sample_pair_groups(groups, target_record_count, rng):
    groups = list(groups)
    group_count = max(1, int(math.ceil(target_record_count / 2)))
    selected_groups = sample_records(groups, group_count, rng)
    records = [record for group in selected_groups for record in group]
    return records[:target_record_count] if len(records) > target_record_count else records


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    distribution = {}
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        if task == "pairwise":
            records = sample_pair_groups(pools["pairwise_groups"], count, rng)
            distribution[task] = {
                "pair_group_pool": len(pools["pairwise_groups"]),
                "record_pool": len(pools["pairwise"]),
                "sampled": len(records),
                "sampled_groups": int(math.ceil(len(records) / 2)),
            }
        else:
            records = sample_records(pools[task], count, rng)
            distribution[task] = {"pool": len(pools[task]), "sampled": len(records)}
        merged.extend(records)
        print(task, distribution[task])
    rng.shuffle(merged)
    return merged, pools, distribution

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools, task_distribution = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)
quick_eval_rows = int(globals().get("QUICK_EVAL_ROWS", 50))
quick_eval_df = validation_df.sample(n=min(quick_eval_rows, len(validation_df)), random_state=SEED).reset_index(drop=True)

run_config_path = RUN_ROOT / "run_config.json"
if run_config_path.exists():
    with open(run_config_path, "r", encoding="utf-8") as f:
        run_config = json.load(f)
else:
    run_config = {"experiment": "qwen2vl_7b_multitask_bipair_conditional_v1", "run_id": RUN_ID}
run_config.update({
    "resume_run_id": RUN_ID,
    "resume_output_dir": str(OUTPUT_DIR),
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "task_distribution_latest_resume": task_distribution,
    "learning_rate_latest_resume": LEARNING_RATE,
    "max_train_steps_latest_resume": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "seed": SEED,
    "train_rows_latest_resume": len(training_df),
    "validation_rows_latest_resume": len(validation_df),
})
with open(run_config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "task_distribution_resume.json", "w", encoding="utf-8") as f:
    json.dump(task_distribution, f, ensure_ascii=False, indent=2)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))
print("train records:", len(train_records))


In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "The two candidate images are labeled A and B in the presented order.\n"
            "Which image occurs earlier in the story timeline?\n"
            "Answer only A or B."
        )
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nWhich image is the first scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nWhich image is the last scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Order all four images from earliest to latest in the story.\n"
            "Answer only four frame numbers separated by spaces, for example: 1 2 3 4."
        )
    if task_type == "fixed_first":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_first']} is fixed as the first scene.\n"
            "Order the remaining frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    if task_type == "fixed_last":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_last']} is fixed as the last scene.\n"
            "Order the remaining frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    if task_type == "fixed_endpoints":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_first']} is fixed as the first scene.\n"
            f"Frame {example['fixed_last']} is fixed as the last scene.\n"
            "Order the remaining middle frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        label = "A" if example["task_type"] == "pairwise" and idx == 1 else "B" if example["task_type"] == "pairwise" and idx == 2 else str(idx)
        content.append({"type": "text", "text": f"\nImage {label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class PilotCDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class PilotCCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for example in batch:
            texts.append(self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False))
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        encoded["labels"] = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        encoded["task_type"] = task_types
        return encoded


In [ ]:
# 5) Build new 7B LoRA adapter and Trainer
class TaskLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        task_types = inputs.pop("task_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        weights = torch.tensor([TASK_LOSS_WEIGHTS.get(task, 1.0) for task in task_types], dtype=sample_losses.dtype, device=sample_losses.device)
        loss = (sample_losses * weights).mean()
        logs = {}
        for task in sorted(set(task_types)):
            task_mask = torch.tensor([value == task for value in task_types], device=sample_losses.device)
            if task_mask.any():
                logs[f"train_{task}_loss"] = sample_losses[task_mask].mean().detach().float().item()
        if logs:
            self.log(logs)
        return (loss, outputs) if return_outputs else loss


# Free stale objects before loading the 7B base model. This matters when a previous
# cell was interrupted during shard loading in the same Colab runtime.
for _name in ["trainer", "model", "base_model", "processor"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory before model load: {free / (1024 ** 3):.1f} GB free / {total / (1024 ** 3):.1f} GB total")


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

checkpoint_dirs = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)
if not checkpoint_dirs:
    raise FileNotFoundError(f"No checkpoint-* directories found under {OUTPUT_DIR}")
RESUME_CHECKPOINT = checkpoint_dirs[-1]
print("Resuming adapter from:", RESUME_CHECKPOINT)

from peft import PeftModel
model = PeftModel.from_pretrained(base_model, RESUME_CHECKPOINT, is_trainable=True)
model.config.use_cache = False
model.print_trainable_parameters()


class QuickEvalCallback(TrainerCallback):
    def on_save(self, args, state, control, model=None, **kwargs):
        if not RUN_QUICK_EVAL_DURING_TRAIN or model is None:
            return control
        path = EVAL_DIR / "train_quick_eval_by_step.csv"
        row = {"step": int(state.global_step)}
        was_training = model.training
        model.eval()
        try:
            row.update(run_cheap_quick_eval(model, quick_eval_df.head(min(12, len(quick_eval_df)))))
        except Exception as exc:
            row["quick_eval_error"] = repr(exc)
        finally:
            if was_training:
                model.train()
        existing = pd.read_csv(path) if path.exists() else pd.DataFrame()
        pd.concat([existing, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)
        print("quick eval:", row)
        return control


@torch.no_grad()
def generate_task_text(active_model, example, max_new_tokens=16):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(next(active_model.parameters()).device) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    output = processor.tokenizer.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return output.strip()


def parse_ab(text):
    match = re.search(r"\b([AB])\b", str(text).upper())
    if match:
        return match.group(1)
    text = str(text).upper().strip()
    return text[0] if text[:1] in {"A", "B"} else None


def run_cheap_quick_eval(active_model, rows):
    metrics = {
        "quick_order_exact": [],
        "quick_first": [],
        "quick_last": [],
        "quick_pair_forward": [],
        "quick_pair_reverse": [],
        "quick_pair_both_directions": [],
        "quick_fixed_first_exact": [],
        "quick_fixed_last_exact": [],
        "quick_fixed_endpoints_exact": [],
    }
    for _, row in rows.iterrows():
        base = base_record(0, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        metrics["quick_order_exact"].append(float(parse_compact_order(generate_task_text(active_model, item), 4) == order))

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        metrics["quick_first"].append(float(str(order[0]) in generate_task_text(active_model, item)[:4]))

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        metrics["quick_last"].append(float(str(order[-1]) in generate_task_text(active_model, item)[:4]))

        pair_forward_correct = []
        pair_reverse_correct = []
        pair_both_correct = []
        for i, j in PAIR_INDICES:
            a, b = i + 1, j + 1
            fwd = copy.deepcopy(base)
            fwd.update({"task_type": "pairwise", "pair": [a, b], "image_paths": [base["image_paths"][a - 1], base["image_paths"][b - 1]], "target": pair_target_for_order(order, a, b)})
            rev = copy.deepcopy(base)
            rev.update({"task_type": "pairwise", "pair": [b, a], "image_paths": [base["image_paths"][b - 1], base["image_paths"][a - 1]], "target": pair_target_for_order(order, b, a)})
            fwd_ok = parse_ab(generate_task_text(active_model, fwd, max_new_tokens=4)) == fwd["target"]
            rev_ok = parse_ab(generate_task_text(active_model, rev, max_new_tokens=4)) == rev["target"]
            pair_forward_correct.append(float(fwd_ok))
            pair_reverse_correct.append(float(rev_ok))
            pair_both_correct.append(float(fwd_ok and rev_ok))
        metrics["quick_pair_forward"].append(float(np.mean(pair_forward_correct)))
        metrics["quick_pair_reverse"].append(float(np.mean(pair_reverse_correct)))
        metrics["quick_pair_both_directions"].append(float(np.mean(pair_both_correct)))

        first = order[0]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order([x for x in order if x != first])})
        metrics["quick_fixed_first_exact"].append(float(parse_compact_order(generate_task_text(active_model, item), 3) == [x for x in order if x != first]))

        last = order[-1]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order([x for x in order if x != last])})
        metrics["quick_fixed_last_exact"].append(float(parse_compact_order(generate_task_text(active_model, item), 3) == [x for x in order if x != last]))

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        metrics["quick_fixed_endpoints_exact"].append(float(parse_compact_order(generate_task_text(active_model, item), 2) == middle))

    return {key: float(np.mean(values)) if values else np.nan for key, values in metrics.items()}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=None,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
)

trainer = TaskLossTrainer(
    model=model,
    args=training_args,
    train_dataset=PilotCDataset(train_records),
    data_collator=PilotCCollator(processor),
    callbacks=[QuickEvalCallback()],
)


In [ ]:
# 6) Resume training and save final adapter
print("resume_from_checkpoint:", RESUME_CHECKPOINT)
trainer.train(resume_from_checkpoint=str(RESUME_CHECKPOINT))
model.save_pretrained(OUTPUT_DIR / "final_adapter")
processor.save_pretrained(OUTPUT_DIR)
print("saved:", OUTPUT_DIR)


## Next

?? ??? ??? `eval_qwen2vl_7b_multitask_bipair_conditional.ipynb`?? ?? `PILOT_RUN_ID`? ??? checkpoint ??? submission ??? ????.
